# 10d SigLIP 2 features for the NSD images

**Kaggle:** GPU T4, Internet on. **Inputs:** the output of 10a.

All 70,566 NSD images through SigLIP 2 grid tokens (layers 4, 8, 12 as 7 x 7 grids), for the comparison of feature
sets with AlexNet and DINOv2. The same kernel pipeline and fit images as 10b. No fMRI is read.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '10d-nsdfeatures', 'gpu-build'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"models": ["siglip2"], "fit_images": 16540, "n_components": 1000, "seed": 344, "alexnet_chunk": 2048, "dinov2_chunk": 4096, "dinov2_layers": [4, 8, 12], "dinov2_pool": 7, "batch": 128}')
TAG = '10d'

import numpy as np
import torch
from cross_subject_encoding.utils import config as C
if not torch.cuda.is_available():
    raise RuntimeError("this notebook needs the GPU accelerator")

In [ ]:
from cross_subject_encoding.features import modern as mf
from cross_subject_encoding.features import streamed
from cross_subject_encoding.models import paper_linearizing as pl
DEVICE = "cuda"
images_dir = next(p for p in Path("/kaggle/input").rglob("images") if p.is_dir() and any(p.glob("nsd-*.jpg")))
conv_dir = images_dir.parent
paths = sorted(images_dir.glob("nsd-*.jpg"))
ids = np.array([int(p.stem.split("-")[1]) for p in paths])
shared = np.load(conv_dir / "shared_train_nsd_ids.npy")
pool = np.where(~np.isin(ids, shared))[0]
fit_idx = np.sort(np.random.default_rng(FIXED["seed"]).choice(pool, FIXED["fit_images"], replace=False))
print(len(paths), "images;", len(shared), "shared (kept out of the fit);", len(fit_idx), "fit images |", f"{(time.monotonic() - T0) / 60:.1f} min")
str_paths = [str(p) for p in paths]
report = {"images": len(paths), "shared": int(len(shared)), "fit_images": int(len(fit_idx)), "models": {}}

def log_progress(name):
    def f(done, total):
        if done == total or done % (10 * FIXED["alexnet_chunk"]) < FIXED["alexnet_chunk"]:
            print(f"  {name}: {done}/{total} |", f"{(time.monotonic() - T0) / 60:.1f} min")
    return f

def modern(name):
    return lambda rows: mf.extract([str_paths[r] for r in rows], name, DEVICE, None, 64, tuple(FIXED["dinov2_layers"]), FIXED["dinov2_pool"])
extractors = {name: ((lambda rows: pl.extract([str_paths[r] for r in rows], DEVICE, batch=FIXED["batch"], workers=2)[0], FIXED["alexnet_chunk"])
                     if name == "alexnet" else (modern(name), FIXED["dinov2_chunk"])) for name in FIXED["models"]}
for name, (fn, chunk) in extractors.items():
    t = time.monotonic()
    torch.cuda.reset_peak_memory_stats()
    z, info = streamed.streamed_kpca(fn, len(paths), fit_idx, DEVICE, FIXED["n_components"], chunk, progress=log_progress(name))
    if not np.isfinite(z).all():
        raise ValueError(f"{name}: non-finite features")
    np.savez(C.PROJECT_ROOT / f"{TAG}_{name}_kpca1000.npz", nsd_ids=ids, z=z, fit_idx=fit_idx)
    report["models"][name] = {**info, "peak_gpu_gib": round(torch.cuda.max_memory_allocated() / 2**30, 2)}
    torch.cuda.empty_cache()
    print(name, report["models"][name])
(C.METRICS_DIR / f"{TAG}_report.json").write_text(json.dumps(report, indent=2))
print(f"{TAG.upper()}_REPORT_JSON " + json.dumps(report))